In [8]:
# ... existing cells ...

# %% [code]
import pandas as pd
from pathlib import Path

csv_path = Path("/Users/ruyi/Desktop/NAIP/v1_resource/NAIDv1/TKPD.csv")  # 替换为实际 CSV 路径
df = pd.read_csv(csv_path)

field_series = df["field"].fillna("").astype(str)
word_counts = field_series.str.split().str.len()
char_lengths = field_series.str.len()

duplicate_counts = field_series.value_counts()
duplicate_only = duplicate_counts[duplicate_counts > 1]

summary = {
    "max_word_count": word_counts.max(),
    "min_word_count": word_counts.min(),
    "char_length_max": char_lengths.max(),
    "char_length_min": char_lengths.min(),
    "unique_keyword_count": int(duplicate_counts.shape[0]),
    "duplicate_keyword_count": int((duplicate_counts > 1).sum()),
}


display(pd.Series(summary, name="field_stats"))

print("\n示例行：")
print(f"单词数最多 ({summary['max_word_count']}): {field_series.iloc[word_counts.idxmax()]}")
print(f"单词数最少 ({summary['min_word_count']}): {field_series.iloc[word_counts.idxmin()]}")
print(f"字符最长 ({summary['char_length_max']}): {field_series.iloc[char_lengths.idxmax()]}")
print(f"字符最短 ({summary['char_length_min']}): {field_series.iloc[char_lengths.idxmin()]}")

print("\n出现次数最多的关键词（Top 10）:")
display(duplicate_counts.head(10).rename("count"))


# ... rest of notebook ...

max_word_count               6
min_word_count               1
char_length_max             47
char_length_min             11
unique_keyword_count       244
duplicate_keyword_count      7
Name: field_stats, dtype: int64


示例行：
单词数最多 (6): Cognitive Ability Evaluation of Language Models
单词数最少 (1): Super-Resolution
字符最长 (47): Cognitive Ability Evaluation of Language Models
字符最短 (11): text-to-SQL

出现次数最多的关键词（Top 10）:


field
language models reasoning     2
neural rendering              2
semantic change detection     2
Text generation               2
fake news detection           2
monocular depth estimation    2
question answering            2
Neural Architecture Search    1
model inversion attacks       1
parasitic egg detection       1
Name: count, dtype: int64

In [9]:
# ... existing cells ...

# %% [code]
import pandas as pd
from pathlib import Path

csv_path = Path("/Users/ruyi/Desktop/NAIP/dataset/arxiv_cs_metadata_keywords.csv")  # 替换为实际 CSV 路径
df = pd.read_csv(csv_path)

field_series = df["predicted_keyword"].fillna("").astype(str)
word_counts = field_series.str.split().str.len()
char_lengths = field_series.str.len()

duplicate_counts = field_series.value_counts()
duplicate_only = duplicate_counts[duplicate_counts > 1]

summary = {
    "max_word_count": word_counts.max(),
    "min_word_count": word_counts.min(),
    "char_length_max": char_lengths.max(),
    "char_length_min": char_lengths.min(),
    "unique_keyword_count": int(duplicate_counts.shape[0]),
    "duplicate_keyword_count": int((duplicate_counts > 1).sum()),
}


display(pd.Series(summary, name="field_stats"))

print("\n示例行：")
print(f"单词数最多 ({summary['max_word_count']}): {field_series.iloc[word_counts.idxmax()]}")
print(f"单词数最少 ({summary['min_word_count']}): {field_series.iloc[word_counts.idxmin()]}")
print(f"字符最长 ({summary['char_length_max']}): {field_series.iloc[char_lengths.idxmax()]}")
print(f"字符最短 ({summary['char_length_min']}): {field_series.iloc[char_lengths.idxmin()]}")

print("\n出现次数最多的关键词（Top 10）:")
display(duplicate_counts.head(10).rename("count"))


# ... rest of notebook ...

max_word_count               10
min_word_count                1
char_length_max              68
char_length_min               4
unique_keyword_count       6383
duplicate_keyword_count    1108
Name: field_stats, dtype: int64


示例行：
单词数最多 (10): evaluation of attitudes opinions and values in large language models
单词数最少 (1): grpc
字符最长 (68): evaluation of attitudes opinions and values in large language models
字符最短 (4): grpc

出现次数最多的关键词（Top 10）:


predicted_keyword
reinforcement learning        94
knowledge distillation        61
contrastive learning          60
self-supervised learning      59
federated learning            49
graph neural networks         42
domain adaptation             39
continual learning            39
medical image segmentation    39
in-context learning           37
Name: count, dtype: int64

In [18]:
# ... existing cells ...

# %% [code]
import numpy as np
import pandas as pd

# 可根据需要调整阈值
min_words = 1
max_words = 6
min_chars = 4
max_chars = 50
max_special_ratio = 0.3  # 特殊字符占比阈值

keywords_df = pd.DataFrame(
    {
        "keyword": field_series,
        "word_count": word_counts,
        "char_length": char_lengths,
    }
)

keywords_df["contains_digits"] = keywords_df["keyword"].str.contains(r"\d")
keywords_df["non_alpha_ratio"] = (
    keywords_df["keyword"]
    .str.replace(r"[A-Za-z\s]", "", regex=True)
    .str.len()
    .div(keywords_df["char_length"].replace(0, np.nan))
    .fillna(0)
)

keywords_df["has_repeated_word"] = (
    keywords_df["keyword"]
    .str.lower()
    .str.split()
    .apply(lambda tokens: len(tokens) != len(set(tokens)) if tokens else False)
)

flag_map = {
    "flag_too_few_words": keywords_df["word_count"] < min_words,
    "flag_too_many_words": keywords_df["word_count"] > max_words,
    "flag_too_short_chars": keywords_df["char_length"] < min_chars,
    "flag_too_long_chars": keywords_df["char_length"] > max_chars,
    # "flag_contains_digits": keywords_df["contains_digits"],
    "flag_high_special_ratio": keywords_df["non_alpha_ratio"] > max_special_ratio,
    "flag_repeated_word": keywords_df["has_repeated_word"],
}

for col, mask in flag_map.items():
    keywords_df[col] = mask

flag_columns = [col for col in keywords_df.columns if col.startswith("flag_")]
suspicious_keywords = keywords_df.loc[keywords_df[flag_columns].any(axis=1)]

cleaned_keywords = (
    keywords_df.loc[~keywords_df[flag_columns].any(axis=1), "keyword"]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"原始关键词数：{len(field_series)}")
print(f"标记为可疑的关键词数：{len(suspicious_keywords)}")
print(f"清洗后剩余关键词数：{len(cleaned_keywords)}")

display(suspicious_keywords.head(20))
cleaned_keywords.head()

# ... rest of notebook ...

原始关键词数：10329
标记为可疑的关键词数：34
清洗后剩余关键词数：6349


,keyword,word_count,char_length,contains_digits,non_alpha_ratio,has_repeated_word,flag_too_few_words,flag_too_many_words,flag_too_short_chars,flag_too_long_chars,flag_high_special_ratio,flag_repeated_word
460,large language models for mental health counse...,7,50,False,0.000000,False,False,True,False,False,False,False
641,code-switching automatic speech recognition me...,5,51,False,0.019608,False,False,False,False,True,False,False
771,reinforcement learning from language model fee...,6,51,False,0.000000,False,False,False,False,True,False,False
920,land use land cover classification,5,34,False,0.000000,True,False,False,False,False,False,True
1015,distributional multi-objective reinforcement l...,4,53,False,0.018868,False,False,False,False,True,False,False
1061,large language model interpretable machine lea...,6,51,False,0.000000,False,False,False,False,True,False,False
1963,electric vehicle charging station location opt...,6,55,False,0.000000,False,False,False,False,True,False,False
2098,reinforcement learning for explainable questio...,6,57,False,0.000000,False,False,False,False,True,False,False
2296,adversarially robust unsupervised domain adapt...,5,51,False,0.000000,False,False,False,False,True,False,False
2357,natural language processing for legal decision...,6,53,False,0.018868,False,False,False,False,True,False,False


0    meta-learning for text-to-speech
1               procedural generation
2          interactive voice response
3                brain age prediction
4    physics-informed neural networks
Name: keyword, dtype: object

In [19]:
import numpy as np
import pandas as pd
from pathlib import Path

# ... existing code ...

metadata_candidates = {
    "arxiv_id": ["arxiv_id"],
    "title": ["title"],
    "abstract": ["abstract"],
}

def _first_available(options):
    for option in options:
        if option in df.columns:
            return option
    return None

metadata_map = {key: _first_available(opts) for key, opts in metadata_candidates.items()}
available_metadata = [col for col in metadata_map.values() if col is not None]

reason_labels = {
    "flag_too_few_words": "too few words",
    "flag_too_many_words": "too many words",
    "flag_too_short_chars": "too few characters",
    "flag_too_long_chars": "too many characters",
    "flag_contains_digits": "contains digits",
    "flag_high_special_ratio": "high special-character ratio",
    "flag_repeated_word": "repeated words",
}

suspicious_keywords = suspicious_keywords.assign(
    suspicious_reasons=suspicious_keywords[flag_columns].apply(
        lambda row: "; ".join(reason_labels[col] for col in flag_columns if row[col]) or "unspecified",
        axis=1,
    )
)

if available_metadata:
    suspicious_with_meta = (
        suspicious_keywords
        .join(df[available_metadata], how="left")
        .rename(columns={col: key for key, col in metadata_map.items() if col})
    )
else:
    suspicious_with_meta = suspicious_keywords.copy()

export_columns = [
    "arxiv_id",
    "title",
    "abstract",
    "keyword",
    "word_count",
    "char_length",
    "suspicious_reasons",
]
existing_export_columns = [col for col in export_columns if col in suspicious_with_meta.columns]
suspicious_export = suspicious_with_meta[existing_export_columns + flag_columns]

output_path = Path("/Users/ruyi/Desktop/NAIP/dataset/suspicious_keywords.csv")
suspicious_export.to_csv(output_path, index=False)
print(f"\n可疑关键词已导出到：{output_path.resolve()}")

display(suspicious_export.head(20))

# ... rest of notebook ...


可疑关键词已导出到：/Users/ruyi/Desktop/NAIP/dataset/suspicious_keywords.csv


,arxiv_id,title,abstract,keyword,word_count,char_length,suspicious_reasons,flag_too_few_words,flag_too_many_words,flag_too_short_chars,flag_too_long_chars,flag_high_special_ratio,flag_repeated_word
460,2311.13857v1,Challenges of Large Language Models for Mental...,The global mental health crisis is looming wit...,large language models for mental health counse...,7,50,too many words,False,True,False,False,False,False
641,2211.16319v1,Benchmarking Evaluation Metrics for Code-Switc...,Code-switching poses a number of challenges an...,code-switching automatic speech recognition me...,5,51,too many characters,False,False,False,True,False,False
771,2401.07382v2,Beyond Sparse Rewards: Enhancing Reinforcement...,Reinforcement learning (RL) can align language...,reinforcement learning from language model fee...,6,51,too many characters,False,False,False,True,False,False
920,2210.14624v1,RapidAI4EO: Mono- and Multi-temporal Deep Lear...,"In the remote sensing community, Land Use Land...",land use land cover classification,5,34,repeated words,False,False,False,False,False,True
1015,2207.00368v1,Multi-Objective Coordination Graphs for the Ex...,Many real-world problems contain multiple obje...,distributional multi-objective reinforcement l...,4,53,too many characters,False,False,False,True,False,False
1061,2402.14474v1,Data Science with LLMs and Interpretable Models,Recent years have seen important advances in t...,large language model interpretable machine lea...,6,51,too many characters,False,False,False,True,False,False
1963,2312.05980v1,Maximum flow-based formulation for the optimal...,"With the increasing effects of climate change,...",electric vehicle charging station location opt...,6,55,too many characters,False,False,False,True,False,False
2098,2210.17095v1,RLET: A Reinforcement Learning Based Approach ...,Interpreting the reasoning process from questi...,reinforcement learning for explainable questio...,6,57,too many characters,False,False,False,True,False,False
2296,2402.11120v1,DART: A Principled Approach to Adversarially R...,Distribution shifts and adversarial examples a...,adversarially robust unsupervised domain adapt...,5,51,too many characters,False,False,False,True,False,False
2357,2308.11531v2,Empowering Refugee Claimants and their Lawyers...,Our project aims at helping and supporting sta...,natural language processing for legal decision...,6,53,too many characters,False,False,False,True,False,False
